# Lab 4 — The Evaluation Harness
**SDA-AIE-111 · Day 3 · Hour 3 · 50 minutes · pairs**

**Objective.** Deliver a cross-validated comparison of all models so far on **shared folds**; tune the threshold to the voucher budget; produce a learning-curve verdict and two error-analysis findings with proposed fixes.

**The rule this lab installs:** every number on the scoreboard comes from `evaluate()` on the shared `CV` object, or it does not go on the scoreboard.


In [ ]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

In [ ]:
from sklearn.model_selection import train_test_split
from manafeth.features import (build_preprocessor, build_feature_table,
                               MODEL_NUM_COLS, CAT_COLS, LEAKY)
from manafeth.evaluation import CV, evaluate, recall_at_budget, precision_at_budget

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
orders = pd.read_parquet(DATA / "manafeth_orders.parquet")
full = build_feature_table(customers, orders, pd.Timestamp("2025-11-01"))

X_all = full.drop(columns=["churned_30d", "customer_id", *LEAKY])
y_all = full["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)
print("train:", X_train.shape, "| churn:", round(y_train.mean(), 3))

## Task 1 — The harness *(10 min)*

`src/manafeth/evaluation.py` defines `evaluate()`: one function, every model, same folds. Comparisons stay legal. Run it on dummy, logistic, and the depth-capped tree.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

def make_pipe(model):
    return Pipeline([("prep", build_preprocessor(MODEL_NUM_COLS, CAT_COLS)), ("clf", model)])

# TODO: build dummy / logistic / tree(depth=6) pipelines and run evaluate() on each
# TODO: tabulate pr_auc_mean ± pr_auc_std; which comparisons show real daylight?


## Task 2 — The threshold is a policy *(10 min)*

Out-of-fold probabilities via `cross_val_predict` — every score is honest. The budget constraint (contact 20% of the base) picks the threshold, not the 0.5 default nobody chose.


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

# TODO: out-of-fold probabilities with cross_val_predict(..., method="predict_proba")
# TODO: plot precision & recall vs threshold; mark 0.5 and the top-20% budget threshold
# TODO: report precision/recall at the budget threshold AND at 0.5 — compare contact rates


## Task 3 — Learning curve: should we buy more data? *(10 min)*

In [ ]:
from sklearn.model_selection import learning_curve

# TODO: learning_curve on the logistic pipeline (train_sizes 10%..100%, shared CV)
# TODO: plot train & validation curves with a std band
# TODO: write the one-sentence verdict: "more data would / would not help because ..."


## Task 4 — Slice-based error analysis *(15 min)*

Where does the model fail, and on whom? Slice recall at the budget threshold by city, tenure band, and device; then read the 20 worst false positives and 20 worst false negatives **row by row**. Findings must come with *proposed fixes* — error analysis that ends at numbers is spectating.


In [ ]:
analysis = X_train.assign(y=y_train.values, proba=proba, flagged=(proba >= budget_thr))

# TODO: implement slice_recall(df, col): recall among churners per group at the budget threshold
# TODO: slice by city, tenure band (0-3 / 3-12 / 12+ months), and device
# TODO: which slice underperforms, and what feature would fix it?


In [ ]:
# TODO: extract the 20 worst false positives (y=0, highest proba) and
#       the 20 worst false negatives (y=1, lowest proba); read them row by row.
# TODO: what do the worst FPs have in common? (hint: look at days_since_last_order
#       together with basket_trend == 1.0 — what does that combination actually mean?)


**Findings (fill in — each with a proposed fix):**

1. Finding: … → Proposed fix: …
2. Finding: … → Proposed fix: …

**Commit:** `feat(lab4): CV harness, budget threshold, error-analysis findings`
